# 🫁 PulmoScan AI — LUNA-22 (ISMI) Training Notebook

**Binary Classification: Benign vs Malignant Lung Nodules**

- 1,176 pre-extracted 3D patches (128×128×64, `.nii.gz`)
- LightweightUNet3D with Depthwise Separable Convolutions (~155K params)
- CombinedLoss (BCE + Dice) · AMP · Cosine LR · Safe Dice metric

In [ ]:
import os, glob, time, warnings
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION — Auto-discovers data on Kaggle
# ═══════════════════════════════════════════════════════════

DATA_DIR = None
METADATA_PATH = None

# Auto-discover: search recursively under /kaggle/input/
for meta in glob.glob('/kaggle/input/**/LIDC-IDRI_1176.npy', recursive=True):
    METADATA_PATH = meta
    break
for nii in glob.glob('/kaggle/input/**/*.nii.gz', recursive=True):
    DATA_DIR = os.path.dirname(nii)
    break

# Local fallback
if not METADATA_PATH: METADATA_PATH = 'data/Dataset/LIDC-IDRI_1176.npy'
if not DATA_DIR: DATA_DIR = 'data/Dataset/LIDC-IDRI'

assert os.path.exists(METADATA_PATH), f'Metadata not found: {METADATA_PATH}'
assert os.path.exists(DATA_DIR), f'Data dir not found: {DATA_DIR}'

BATCH_SIZE   = 16
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-4
TARGET_SHAPE = (64, 64, 32)
TEST_SPLIT   = 0.2
SEED         = 42
NUM_WORKERS  = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

nii_count = len([f for f in os.listdir(DATA_DIR) if f.endswith('.nii.gz')])
print(f'Metadata:  {METADATA_PATH}')
print(f'Data dir:  {DATA_DIR} ({nii_count} files)')
print(f'Device:    {DEVICE}')
print(f'Config:    EPOCHS={EPOCHS}, LR={LR}, BS={BATCH_SIZE}')

In [ ]:
metadata = np.load(METADATA_PATH, allow_pickle=True)
print(f'Metadata entries: {len(metadata)}')
print(f'Keys: {list(metadata[0].keys())}')
print(f'Sample: {metadata[0]}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# BUILD FILE PATHS + BINARY LABELS
# ═══════════════════════════════════════════════════════════

actual_files = set(os.listdir(DATA_DIR))
file_paths, labels, skipped = [], [], 0

for entry in metadata:
    fname = entry['Filename']
    if fname in actual_files:
        fpath = os.path.join(DATA_DIR, fname)
    else:
        # Fuzzy match (handles minor naming differences)
        matches = [f for f in actual_files if fname in f or f in fname]
        if matches:
            fpath = os.path.join(DATA_DIR, matches[0])
        else:
            skipped += 1
            continue
    
    mal = np.median(entry['Malignancy'])
    file_paths.append(fpath)
    labels.append(1 if mal >= 3 else 0)

labels = np.array(labels)
print(f'Matched: {len(file_paths)} | Skipped: {skipped}')
print(f'Benign: {(labels==0).sum()} | Malignant: {(labels==1).sum()}')
assert len(file_paths) > 0, 'ERROR: No samples found!'

In [ ]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, labels, test_size=TEST_SPLIT, random_state=SEED, stratify=labels
)
print(f'Train: {len(train_paths)} (B:{(np.array(train_labels)==0).sum()}, M:{(np.array(train_labels)==1).sum()})')
print(f'Test:  {len(test_paths)} (B:{(np.array(test_labels)==0).sum()}, M:{(np.array(test_labels)==1).sum()})')

In [ ]:
class LUNA22Dataset(Dataset):
    def __init__(self, paths, labels, shape=(64,64,32), augment=False):
        self.paths, self.labels, self.shape, self.augment = paths, labels, shape, augment
    
    def __len__(self): return len(self.paths)
    
    def __getitem__(self, idx):
        try:
            vol = nib.load(self.paths[idx]).get_fdata().astype(np.float32)
        except Exception:
            vol = np.zeros(self.shape, dtype=np.float32)
        
        # Normalize [0, 1]
        mn, mx = vol.min(), vol.max()
        vol = (vol - mn) / (mx - mn + 1e-8) if mx - mn > 1e-8 else np.zeros_like(vol)
        
        # Resize
        if vol.shape != self.shape:
            t = torch.from_numpy(vol).float().unsqueeze(0).unsqueeze(0)
            vol = F.interpolate(t, size=self.shape, mode='trilinear', align_corners=False).squeeze().numpy()
        
        # Augment
        if self.augment:
            for ax in range(3):
                if np.random.random() > 0.5: vol = np.flip(vol, axis=ax).copy()
        
        return torch.from_numpy(vol).float().unsqueeze(0), torch.tensor([self.labels[idx]], dtype=torch.float32)

train_ds = LUNA22Dataset(train_paths, train_labels, TARGET_SHAPE, augment=True)
test_ds  = LUNA22Dataset(test_paths, test_labels, TARGET_SHAPE, augment=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_dl  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

x, y = train_ds[0]
print(f'Sample: vol={x.shape}, label={y.item()}, range=[{x.min():.3f}, {x.max():.3f}]')

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL (must match src/models/lightweight_unet.py exactly)
# ═══════════════════════════════════════════════════════════

class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_ch, in_ch, kernel_size=kernel_size, stride=stride, padding=padding, groups=in_ch, bias=False)
        self.pointwise = nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm3d(out_ch)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu(self.bn(self.pointwise(self.depthwise(x))))

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv3d(in_ch, out_ch)
        self.conv2 = DepthwiseSeparableConv3d(out_ch, out_ch)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
    def forward(self, x):
        x = self.conv1(x); x = self.conv2(x)
        return x, self.pool(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv1 = DepthwiseSeparableConv3d(out_ch * 2, out_ch)
        self.conv2 = DepthwiseSeparableConv3d(out_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape: x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv2(self.conv1(x))

class LightweightUNet3D(nn.Module):
    def __init__(self, in_channels=1, features=None):
        super().__init__()
        if features is None: features = [16, 32, 64]
        self.enc1 = EncoderBlock(in_channels, features[0])
        self.enc2 = EncoderBlock(features[0], features[1])
        self.enc3 = EncoderBlock(features[1], features[2])
        self.bottleneck = nn.Sequential(
            DepthwiseSeparableConv3d(features[2], features[2]*2),
            DepthwiseSeparableConv3d(features[2]*2, features[2]*2),
        )
        self.dec3 = DecoderBlock(features[2]*2, features[2])
        self.dec2 = DecoderBlock(features[2], features[1])
        self.dec1 = DecoderBlock(features[1], features[0])
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(features[0], 1),
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm3d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.constant_(self.classifier[-1].bias, -5.0)

    def forward(self, x):
        s1, x = self.enc1(x); s2, x = self.enc2(x); s3, x = self.enc3(x)
        x = self.bottleneck(x)
        x = self.dec3(x, s3); x = self.dec2(x, s2); x = self.dec1(x, s1)
        return self.classifier(x)

model = LightweightUNet3D().to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,} (~{params*4/1024/1024:.2f} MB)')

In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, bce_w=0.7, dice_w=0.3, pos_weight=None):
        super().__init__()
        self.bce_w, self.dice_w = bce_w, dice_w
        pw = torch.tensor([pos_weight]) if pos_weight else None
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    def dice_loss(self, logits, targets):
        p = torch.sigmoid(logits); s = 1e-6
        return 1 - (2*(p*targets).sum()+s) / (p.sum()+targets.sum()+s)
    def forward(self, logits, targets):
        return self.bce_w * self.bce(logits, targets) + self.dice_w * self.dice_loss(logits, targets)

n_b = (np.array(train_labels)==0).sum()
n_m = (np.array(train_labels)==1).sum()
pw = n_b / max(n_m, 1)
print(f'Pos weight: {pw:.2f} ({n_b} benign / {n_m} malignant)')
criterion = CombinedLoss(pos_weight=pw).to(DEVICE)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler('cuda') if DEVICE.type == 'cuda' else None
print(f'Optimizer: AdamW(lr={LR}), Scheduler: Cosine(T={EPOCHS}), AMP: {scaler is not None}')

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    loss_sum, preds, lbls = 0, [], []
    for data, targets in loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        if scaler:
            with autocast('cuda'):
                out = model(data); loss = criterion(out, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            out = model(data); loss = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        loss_sum += loss.item()
        preds.extend((torch.sigmoid(out)>0.5).float().cpu().numpy().flatten())
        lbls.extend(targets.cpu().numpy().flatten())
    return loss_sum/len(loader), accuracy_score(lbls, preds)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, preds, lbls, probs = 0, [], [], []
    pos_p, pos_l = [], []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            if device.type=='cuda':
                with autocast('cuda'): out = model(data); loss = criterion(out, targets)
            else: out = model(data); loss = criterion(out, targets)
            loss_sum += loss.item()
            p = torch.sigmoid(out)
            preds.extend((p>0.5).float().cpu().numpy().flatten())
            lbls.extend(targets.cpu().numpy().flatten())
            probs.extend(p.cpu().numpy().flatten())
            mask = targets.cpu().numpy().flatten() == 1
            if mask.any():
                pos_p.extend(p.cpu().numpy().flatten()[mask])
                pos_l.extend(targets.cpu().numpy().flatten()[mask])
    lbls, preds = np.array(lbls), np.array(preds)
    dice = 0.0
    if pos_p:
        pp, pl = np.array(pos_p), np.array(pos_l)
        dice = (2*(pp*pl).sum()+1e-6) / (pp.sum()+pl.sum()+1e-6)
    return {
        'loss': loss_sum/len(loader),
        'accuracy': accuracy_score(lbls, preds),
        'precision': precision_score(lbls, preds, zero_division=0),
        'recall': recall_score(lbls, preds, zero_division=0),
        'f1': f1_score(lbls, preds, zero_division=0),
        'dice_pos': dice
    }

In [ ]:
best_f1 = 0.0
hist = {'tl':[], 'ta':[], 'vl':[], 'va':[], 'vf':[], 'vd':[], 'lr':[]}

print('='*80)
print(f'{"Ep":>4} | {"TrLoss":>8} | {"VaLoss":>8} | {"Acc":>6} | {"F1":>6} | {"Dice+":>6} | {"LR":>10}')
print('='*80)

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    tl, ta = train_epoch(model, train_dl, criterion, optimizer, scaler, DEVICE)
    vm = validate_epoch(model, test_dl, criterion, DEVICE)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    hist['tl'].append(tl); hist['ta'].append(ta)
    hist['vl'].append(vm['loss']); hist['va'].append(vm['accuracy'])
    hist['vf'].append(vm['f1']); hist['vd'].append(vm['dice_pos']); hist['lr'].append(lr)
    
    print(f'{ep:>4} | {tl:>8.4f} | {vm["loss"]:>8.4f} | {vm["accuracy"]:>5.1%} | {vm["f1"]:>6.3f} | {vm["dice_pos"]:>6.3f} | {lr:>10.2e} | {time.time()-t0:.1f}s')
    
    if vm['f1'] > best_f1:
        best_f1 = vm['f1']
        torch.save({'epoch': ep, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_f1': best_f1, 'val_metrics': vm,
                    'config': {'target_shape': TARGET_SHAPE, 'features': [16,32,64],
                               'dataset': 'LUNA-22 (ISMI)', 'task': 'binary_classification'}
                   }, 'lung_nodule_net_best.pth')
        print(f'      ✅ Best model saved (F1={best_f1:.4f})')

print(f'\nDone. Best F1: {best_f1:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training History — LUNA-22 LightweightUNet3D', fontsize=14, fontweight='bold')
axes[0,0].plot(hist['tl'], label='Train', color='#2196F3'); axes[0,0].plot(hist['vl'], label='Val', color='#FF5722')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(hist['ta'], label='Train', color='#2196F3'); axes[0,1].plot(hist['va'], label='Val', color='#FF5722')
axes[0,1].set_title('Accuracy'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)
axes[1,0].plot(hist['vf'], label='F1', color='#4CAF50'); axes[1,0].plot(hist['vd'], label='Dice+', color='#9C27B0', ls='--')
axes[1,0].set_title('F1 & Safe Dice'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(hist['lr'], color='#FF9800'); axes[1,1].set_title('Learning Rate'); axes[1,1].set_yscale('log'); axes[1,1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
ckpt = torch.load('lung_nodule_net_best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best model from epoch {ckpt["epoch"]}')

fm = validate_epoch(model, test_dl, criterion, DEVICE)
print(f'\n{"="*50}')
print(f'  Accuracy:   {fm["accuracy"]:.1%}')
print(f'  Precision:  {fm["precision"]:.4f}')
print(f'  Recall:     {fm["recall"]:.4f}')
print(f'  F1:         {fm["f1"]:.4f}')
print(f'  Dice (pos): {fm["dice_pos"]:.4f}')
print(f'{"="*50}')

In [ ]:
model.eval()
all_p, all_l = [], []
with torch.no_grad():
    for d, t in test_dl:
        d = d.to(DEVICE)
        all_p.extend((torch.sigmoid(model(d))>0.5).float().cpu().numpy().flatten())
        all_l.extend(t.numpy().flatten())

cm = confusion_matrix(all_l, all_p)
print('Confusion Matrix:')
print(f'              Predicted')
print(f'              Benign  Malignant')
print(f'  Benign       {cm[0,0]:>4}     {cm[0,1]:>4}')
print(f'  Malignant    {cm[1,0]:>4}     {cm[1,1]:>4}')
print(f'\n{classification_report(all_l, all_p, target_names=["Benign", "Malignant"])}')

In [ ]:
sz = os.path.getsize('lung_nodule_net_best.pth') / 1024
print(f'\n📦 Model: lung_nodule_net_best.pth ({sz:.1f} KB)')
print(f'   Params: {sum(p.numel() for p in model.parameters()):,}')
print(f'\n🎯 Next steps:')
print(f'   1. Download lung_nodule_net_best.pth from Kaggle Output')
print(f'   2. Place in project root directory')
print(f'   3. Run: python src/api/server.py')